## PCA por Métricas

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from scipy.stats import ttest_ind

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [2]:
DATASET_PATH = "../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape

(165, 393)

In [16]:
#  Age, Gender and condition
# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

NODE_FEATURES_BY_TYPE = {
    prefix.rstrip("_"): [c for c in NODE_FEATURES if c.startswith(prefix)]
    for prefix in NODE_FEATURE_PREFIXES
}

NODE_GROUPS = {
    "degree": [c for c in df_dataset.columns if c.startswith("degree_")],
    "clustering": [c for c in df_dataset.columns if c.startswith("clustering_")],
    "betweenness": [c for c in df_dataset.columns if c.startswith("betweenness_")],
    "local_efficiency": [c for c in df_dataset.columns if c.startswith("local_efficiency_")],
    "weighted_degree": [c for c in df_dataset.columns if c.startswith("weighted_degree_")]
}

In [17]:
def plot_pca_features(df, features, title="PCA"):
    # Filter data
    X = df[features].values
    X = np.nan_to_num(X, nan=0.0)
    
    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # Create DataFrame for plotting
    df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
    
    # Add condition for coloring if available
    if "condition" in df.columns:
        df_pca["condition"] = df["condition"].values
        color_col = "condition"
    else:
        color_col = None
        
    # Plot
    fig = px.scatter(
        df_pca, x="PC1", y="PC2", 
        color=color_col,
        title=title,
        opacity=0.8
    )
    fig.show()
    return pca


Grau

In [18]:
pca_degree = plot_pca_features(
    df_dataset, # Replace with df_dataset if that's your variable name
    NODE_FEATURES_BY_TYPE["degree"],
    title="PCA das Métricas de Grau (Nós)"
)

Clustering

In [19]:
pca_clustering = plot_pca_features(
    df_dataset, 
    NODE_FEATURES_BY_TYPE["clustering"],
    title="PCA das Métricas de Clustering"
)

Centralidade de Intermediação

In [20]:
pca_betweenness = plot_pca_features(
    df_dataset,
    NODE_GROUPS["betweenness"],
    title="PCA da Centralidade de Intermediação (Nós)"
)


Grau ponderado

In [21]:
pca_wdeg = plot_pca_features(
    df_dataset,
    NODE_GROUPS["weighted_degree"],
    title="PCA do Grau Ponderado (Nós)"
)
